<a href="https://colab.research.google.com/github/sakaleshhubli/sem7-ML2-/blob/main/5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Required Libraries
We need to install the `wittgenstein` library, which provides an easy-to-use implementation of the RIPPER (Repeated Incremental Pruning to Produce Error Reduction) algorithm.

In [1]:
!pip install wittgenstein pandas scikit-learn

### 2. Load Dataset and Apply RIPPER
We will use the classic Iris dataset from `scikit-learn`, simplify it into a binary classification task (e.g., setosa vs. non-setosa), and then train a RIPPER classifier to extract human-readable rules.

In [2]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import wittgenstein as lw

# Load dataset
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
# Turn this into a binary classification task: Is it 'setosa'?
df['is_setosa'] = (iris.target == 0).astype(int)

# Split into training and test sets
train, test = train_test_split(df, test_size=0.3, random_state=42)

# Initialize and fit RIPPER model
ripper = lw.RIPPER(random_state=42)
ripper.fit(train, class_feat='is_setosa')

# Display the rules found
print("Generated RIPPER Rules:")
print(ripper.ruleset_)

# Separate features and target for evaluation
X_test = test.drop(columns=['is_setosa'])
y_test = test['is_setosa']

# Evaluate model performance
accuracy = ripper.score(X_test, y_test)
print(f"\nTest Accuracy: {accuracy:.4f}")

Generated RIPPER Rules:
[[petalwidth(cm)=<0.2] V [petalwidth(cm)=0.2-1.0^sepallength(cm)=5.0-5.22] V [petallength(cm)=<1.4] V [petallength(cm)=1.5-3.06]]

Test Accuracy: 0.9556


### 3. FOIL Algorithm (First-Order Inductive Learner)
FOIL is used to learn first-order logic rules (Horn clauses) from relational database facts. Below is a Python implementation representing how FOIL searches for relations using a simplified model of predicates.

In [3]:
# We define a simple database of facts to demonstrate FOIL's logic
# Target predicate: grandparent(X, Y) :- parent(X, Z), parent(Z, Y)

facts = {
    'parent': [
        ('alice', 'bob'),
        ('bob', 'charlie'),
        ('dan', 'eve'),
        ('eve', 'frank')
    ],
    'grandparent': [
        ('alice', 'charlie'),
        ('dan', 'frank')
    ]
}

def foil_gain(pos_before, neg_before, pos_after, neg_after):
    import math
    if pos_after == 0:
        return 0
    info_before = -math.log2(pos_before / (pos_before + neg_before))
    info_after = -math.log2(pos_after / (pos_after + neg_after))
    return pos_after * (info_before - info_after)

# Display example information-gain computation for FOIL literal selection
pos_b, neg_b = 2, 4  # Prior distribution
pos_a, neg_a = 2, 1  # Distribution after adding parent(X, Z)

gain = foil_gain(pos_b, neg_b, pos_a, neg_a)
print(f"FOIL Information Gain for candidate literal: {gain:.4f}")

FOIL Information Gain for candidate literal: 2.0000


In [4]:
# @title 📊 Iris Classification & RIPPER Rules Dashboard {display-mode: "form"}

import json
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import wittgenstein as lw
from IPython.display import HTML
from google.colab import output

# Register error reporting callback
def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

# Prepare data
iris = load_iris()
df_data = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df_data['is_setosa'] = (iris.target == 0).astype(int)

train, test = train_test_split(df_data, test_size=0.3, random_state=42)
ripper = lw.RIPPER(random_state=42)
ripper.fit(train, class_feat='is_setosa')

# Calculate metrics
accuracy = ripper.score(test.drop(columns=['is_setosa']), test['is_setosa'])
total_samples = len(df_data)
setosa_count = int(df_data['is_setosa'].sum())
non_setosa_count = total_samples - setosa_count

# Serialize dataset for frontend
dataset_json = df_data.to_json(orient='records')
rules_str = str(ripper.ruleset_)

html_content = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Iris Classification & Rules Dashboard</title>
    <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
    <style>
        body {
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
            background-color: #f4f6f8;
            color: #333;
            margin: 0;
            padding: 20px;
        }
        .dashboard-container {
            max-width: 1200px;
            margin: 0 auto;
        }
        .header {
            margin-bottom: 24px;
        }
        .header h1 {
            margin: 0 0 8px 0;
            font-size: 28px;
            color: #1a1f36;
        }
        .header p {
            margin: 0;
            color: #4f566b;
        }
        .kpi-row {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
            gap: 16px;
            margin-bottom: 24px;
        }
        .kpi-card {
            background-color: #ffffff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            border: 1px solid #e3e8ee;
        }
        .kpi-card .title {
            font-size: 14px;
            color: #697386;
            margin-bottom: 8px;
            text-transform: uppercase;
            letter-spacing: 0.5px;
        }
        .kpi-card .value {
            font-size: 24px;
            font-weight: bold;
            color: #1a1f36;
        }
        .charts-row {
            display: grid;
            grid-template-columns: 2fr 1fr;
            gap: 24px;
            margin-bottom: 24px;
        }
        @media (max-width: 768px) {
            .charts-row {
                grid-template-columns: 1fr;
            }
        }
        .chart-card {
            background-color: #ffffff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            border: 1px solid #e3e8ee;
            display: flex;            flex-direction: column;
        }
        .chart-card h2 {
            margin: 0 0 16px 0;
            font-size: 18px;
            color: #1a1f36;
        }
        .canvas-wrapper {
            position: relative;
            flex-grow: 1;
            min-height: 250px;
        }
        .rules-card {
            background-color: #ffffff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            border: 1px solid #e3e8ee;
            margin-bottom: 24px;
        }
        .rules-card h2 {
            margin: 0 0 12px 0;
            font-size: 18px;
            color: #1a1f36;
        }
        .rules-box {
            background-color: #f8f9fa;
            border-left: 4px solid #635bff;
            padding: 12px 16px;
            font-family: monospace;
            font-size: 14px;
            white-space: pre-wrap;
            word-break: break-all;
            margin: 0;
        }
        .table-card {
            background-color: #ffffff;
            border-radius: 8px;
            padding: 20px;
            box-shadow: 0 1px 3px rgba(0,0,0,0.1);
            border: 1px solid #e3e8ee;
        }
        .table-card h2 {
            margin: 0 0 16px 0;
            font-size: 18px;
            color: #1a1f36;
        }
        .table-container {
            overflow-x: auto;
        }
        table {
            width: 100%;
            border-collapse: collapse;
            text-align: left;
            font-size: 14px;
        }
        th {
            background-color: #f8f9fa;
            padding: 12px;
            font-weight: 600;
            color: #4f566b;
            border-bottom: 2px solid #e3e8ee;
        }
        td {
            padding: 12px;
            border-bottom: 1px solid #e3e8ee;
            color: #3c4257;
        }
        tr:hover {
            background-color: #f9fafb;
        }
    </style>
</head>
<body>
    <div class="dashboard-container">
        <div class="header">
            <h1>Iris & RIPPER Classifier Dashboard</h1>
            <p>Interactively explore RIPPER rules and underlying dataset metrics</p>
        </div>

        <div class="kpi-row">
            <div class="kpi-card">
                <div class="title">Model Accuracy</div>
                <div class="value">ACCURACY_PLACEHOLDER%</div>
            </div>
            <div class="kpi-card">
                <div class="title">Total Samples</div>
                <div class="value">TOTAL_SAMPLES_PLACEHOLDER</div>
            </div>
            <div class="kpi-card">
                <div class="title">Setosa Instances</div>
                <div class="value">SETOSA_COUNT_PLACEHOLDER</div>
            </div>
            <div class="kpi-card">
                <div class="title">Non-Setosa Instances</div>
                <div class="value">NON_SETOSA_COUNT_PLACEHOLDER</div>
            </div>
        </div>

        <div class="rules-card">
            <h2>Extracted RIPPER Ruleset</h2>
            <pre class="rules-box">RULES_PLACEHOLDER</pre>
        </div>

        <div class="charts-row">
            <div class="chart-card">
                <h2>Petal Measurements Distribution</h2>
                <div class="canvas-wrapper">
                    <canvas id="petalScatterChart"></canvas>
                </div>
            </div>
            <div class="chart-card">
                <h2>Target Distribution</h2>
                <div class="canvas-wrapper">
                    <canvas id="targetPieChart"></canvas>
                </div>
            </div>
        </div>

        <div class="table-card">
            <h2>Dataset Preview (First 10 Rows)</h2>
            <div class="table-container">
                <table id="previewTable">
                    <thead>
                        <tr>
                            <th>Sepal Length (cm)</th>
                            <th>Sepal Width (cm)</th>
                            <th>Petal Length (cm)</th>
                            <th>Petal Width (cm)</th>
                            <th>Is Setosa</th>
                        </tr>
                    </thead>
                    <tbody></tbody>
                </table>
            </div>
        </div>
    </div>

    <script>
        window.onerror = function(message) {
            google.colab.kernel.invokeFunction('report_js_error', [message], {});
        };

        const data = DATA_PLACEHOLDER;

        // Render Table Rows
        const tbody = document.querySelector('#previewTable tbody');
        data.slice(0, 10).forEach(row => {
            const tr = document.createElement('tr');
            tr.innerHTML = `
                <td>${row['sepal length (cm)']}</td>
                <td>${row['sepal width (cm)']}</td>
                <td>${row['petal length (cm)']}</td>
                <td>${row['petal width (cm)']}</td>
                <td><strong>${row['is_setosa'] === 1 ? 'Yes (1)' : 'No (0)'}</strong></td>
            `;
            tbody.appendChild(tr);
        });

        // Target Distribution Chart
        const ctxPie = document.getElementById('targetPieChart').getContext('2d');
        new Chart(ctxPie, {
            type: 'doughnut',
            data: {
                labels: ['Non-Setosa', 'Setosa'],
                datasets: [{
                    data: [NON_SETOSA_VAL, SETOSA_VAL],
                    backgroundColor: ['#e3e8ee', '#635bff'],
                    borderWidth: 1
                }]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                plugins: {
                    legend: { position: 'bottom' }
                }
            }
        });

        // Scatter Chart
        const scatterData = data.map(row => ({
            x: row['petal length (cm)'],
            y: row['petal width (cm)'],
            isSetosa: row['is_setosa']
        }));

        const ctxScatter = document.getElementById('petalScatterChart').getContext('2d');
        new Chart(ctxScatter, {
            type: 'scatter',
            data: {
                datasets: [
                    {
                        label: 'Non-Setosa',
                        data: scatterData.filter(d => d.isSetosa === 0),
                        backgroundColor: '#e3e8ee',
                        pointRadius: 6
                    },
                    {
                        label: 'Setosa',
                        data: scatterData.filter(d => d.isSetosa === 1),
                        backgroundColor: '#635bff',
                        pointRadius: 6
                    }
                ]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                scales: {
                    x: {
                        title: { display: true, text: 'Petal Length (cm)' }
                    },
                    y: {
                        title: { display: true, text: 'Petal Width (cm)' }
                    }
                }
            }
        });
    </script>
</body>
</html>
"""

# Safely replace placeholders
html_rendered = html_content.replace('DATA_PLACEHOLDER', dataset_json) \
                            .replace('ACCURACY_PLACEHOLDER', f"{accuracy*100:.2f}") \
                            .replace('TOTAL_SAMPLES_PLACEHOLDER', str(total_samples)) \
                            .replace('SETOSA_COUNT_PLACEHOLDER', str(setosa_count)) \
                            .replace('NON_SETOSA_COUNT_PLACEHOLDER', str(non_setosa_count)) \
                            .replace('RULES_PLACEHOLDER', rules_str) \
                            .replace('NON_SETOSA_VAL', str(non_setosa_count)) \
                            .replace('SETOSA_VAL', str(setosa_count))

HTML(html_rendered)


Sepal Length (cm),Sepal Width (cm),Petal Length (cm),Petal Width (cm),Is Setosa
